# 04 - Feature Selection (Late Fusion Architecture)

This notebook demonstrates the Late Fusion feature selection pipeline:

**NEW ARCHITECTURE:** Two parallel branches instead of Early Fusion
1. **Genomic Branch:** MI + PSO on gene expression data ONLY
2. **Clinical Branch:** Domain-specific engineered features (Gleason, Stage, etc.) ONLY

> **CRITICAL:** Old 3-Layer/Early Fusion logic has been REMOVED

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import config
from src.io import save_dataframe, save_table, logger
from src.genomic_selector import run_genomic_feature_selection, transform_genomic
from src.clinical_engineer import create_clinical_features, get_clinical_feature_names
from src.visualization import setup_style
setup_style()

## Step 1: Load Preprocessed Data and Split into Genomic/Clinical

In [ ]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv")
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

print(f"Train shape: {X_train.shape}, positives={int(y_train.sum())}")
print(f"Test shape:  {X_test.shape}, positives={int(y_test.sum())}")

In [ ]:
# Identify genomic vs clinical columns
GENE_COLS = [c for c in X_train.columns if c not in [
    'Gleason pattern primary', 'Gleason pattern secondary',
    'Surgical Margin Resection Status_R1',
    'Primary Lymph Node Presentation Assessment Ind-3_YES'
] and not any(x in c for x in ['Tumor Stage Code_', 'pathology'])]

CLINICAL_COLS = [c for c in X_train.columns if c not in GENE_COLS]

X_train_genomic = X_train[GENE_COLS]
X_train_clinical = X_train[CLINICAL_COLS]
X_test_genomic = X_test[GENE_COLS]
X_test_clinical = X_test[CLINICAL_COLS]

print(f"Genomic features: {len(GENE_COLS)}")
print(f"Clinical features: {len(CLINICAL_COLS)}")

## Step 2: Genomic Branch - MI + PSO on Genes ONLY

In [ ]:
genomic_selector, genomic_features = run_genomic_feature_selection(
    X_train_genomic,
    y_train,
    variance_threshold=config.VARIANCE_THRESHOLD,
    mi_top_k=config.MI_TOP_K,
    pso_final_k=config.PSO_FINAL_K,
    run_pso=True,
    random_state=config.RANDOM_STATE,
)

print(f"\nGenomic Branch: Selected {len(genomic_features)} genes")

## Step 3: Clinical Branch - Create Engineered Features ONLY

In [ ]:
X_train_clinical_eng, clinical_features = create_clinical_features(X_train_clinical)
X_test_clinical_eng, _ = create_clinical_features(X_test_clinical)

print(f"Clinical Branch: Created {len(clinical_features)} engineered features")
print(f"Features: {clinical_features}")

## Step 4: Display Selected Genomic Features

In [ ]:
genomic_df = pd.DataFrame({
    "rank": range(1, len(genomic_features) + 1),
    "feature": genomic_features,
})
print("Selected Genomic Features:")
print(genomic_df.to_string(index=False))

## Step 5: Mutual Information Scores (Top Genomic Features)

In [ ]:
mi_scores = genomic_selector["mi_scores"]
top_mi = mi_scores.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top_mi.sort_values().plot(kind="barh", ax=ax, color="#3C5488")
ax.set_xlabel("Mutual Information Score")
ax.set_title("Top 20 Genomic Features by Mutual Information")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()

from src.io import save_figure
save_figure(fig, "genomic_mi_scores.png")
plt.show()

## Step 6: Transform Test Data Using Fitted Selectors

In [ ]:
X_train_genomic_selected = transform_genomic(X_train_genomic, genomic_selector, genomic_features)
X_test_genomic_selected = transform_genomic(X_test_genomic, genomic_selector, genomic_features)

print(f"Train genomic selected shape: {X_train_genomic_selected.shape}")
print(f"Test genomic selected shape:  {X_test_genomic_selected.shape}")

## Step 7: Save Artifacts for Late Fusion

In [ ]:
save_table(pd.DataFrame({"feature": genomic_features}), "genomic_features.csv", index=False)
save_table(pd.DataFrame({"feature": clinical_features}), "clinical_features.csv", index=False)

save_dataframe(X_train_genomic_selected, "X_train_genomic_selected.csv", index=False)
save_dataframe(X_test_genomic_selected, "X_test_genomic_selected.csv", index=False)
save_dataframe(X_train_clinical_eng, "X_train_clinical_engineered.csv", index=False)
save_dataframe(X_test_clinical_eng, "X_test_clinical_engineered.csv", index=False)

print("Saved:")
print(f"  - genomic_features.csv ({len(genomic_features)} genes)")
print(f"  - clinical_features.csv ({len(clinical_features)} features)")
print(f"  - X_train_genomic_selected.csv {X_train_genomic_selected.shape}")
print(f"  - X_test_genomic_selected.csv {X_test_genomic_selected.shape}")
print(f"  - X_train_clinical_engineered.csv {X_train_clinical_eng.shape}")
print(f"  - X_test_clinical_engineered.csv {X_test_clinical_eng.shape}")